In [1]:
from datetime import datetime
from pydantic import BaseModel, PositiveInt, ValidationError

# 터미널에서 dict 구조를 읽기 편하게 하기 위해서.
from pprint import pprint


# id = 정수형
# name = 문자열인데 디폴트 값은 "백효영"
# signup_ts = datetime 타입인데 None이 들어올 수 있음. 인자 생략이 되는건 아님.
# tastes = 키는 문자열, 밸류는 양의정수.
class User(BaseModel):
    id: int
    name: str = "백효영"
    signup_ts: datetime | None
    tastes: dict[str, PositiveInt]


# 외부에서 들어오는 데이터라고 가정하고 오브젝트 하나 만듬.
external_data = {
    "id": "000829",
    "signup_ts": "2026-09-07 10:08",
    "tastes": {
        "wine": 9,
        b"cheese": 7,
        "cabbage": "1",
        "burger": "10",
    },
}

# js의 스프레드 문법과 비슷하지만 살짝 다르다고 함.
# User 클래스에 '외부 데이터'를 주입해서 "검증"된 객체를 만들겠다! 라는 의미.
# 왜? User 클래스는 pydantic를 상속받아 만들어진 클래스임. 그래서 검증 단계가 끼게 됨.
# 외부 데이터의 순서가 바뀌어도 "키"를 바탕으로 매칭하기 때문에 순서는 상관 없다.
# "**"를 사용해서 딕트를 풀어줘야 User 클래스는 필요한 인자를 받을 수 있음.
user = User(**external_data)

# 만들어진 객체의 데이터 호출.
print(user.id)
print(user.model_dump())

829
{'id': 829, 'name': '백효영', 'signup_ts': datetime.datetime(2026, 9, 7, 10, 8), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1, 'burger': 10}}


In [2]:
from datetime import datetime
from pydantic import BaseModel, PositiveInt, ValidationError
from pprint import pprint

class User(BaseModel):
    id: int
    name: str = "백효영"
    signup_ts: datetime | None
    tastes: dict[str, PositiveInt]

error_external_data = {
    "id": "한글이지롱",
    "signup_ts": "이것도 타임스탬프 아니지롱",
    "name": "1029098235",
    "tastes": {"사과": 4, "배": 5, "오렌지": 6, "피자": "맛있어"},
}

# JS의 try-catch 구문과 같음.
# 에러를 마주하면 pydantic의 VaildationError의 메서드를 통해 어디서 어떤 에러가 발생했는지 출력할 수 있다.
try:
    User(**error_external_data)
except ValidationError as e:
    pprint(e.errors())


[{'input': '한글이지롱',
  'loc': ('id',),
  'msg': 'Input should be a valid integer, unable to parse string as an '
         'integer',
  'type': 'int_parsing',
  'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'},
 {'ctx': {'error': 'invalid character in year'},
  'input': '이것도 타임스탬프 아니지롱',
  'loc': ('signup_ts',),
  'msg': 'Input should be a valid datetime or date, invalid character in year',
  'type': 'datetime_from_date_parsing',
  'url': 'https://errors.pydantic.dev/2.13/v/datetime_from_date_parsing'},
 {'input': '맛있어',
  'loc': ('tastes', '피자'),
  'msg': 'Input should be a valid integer, unable to parse string as an '
         'integer',
  'type': 'int_parsing',
  'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}]


In [3]:
from pydantic import BaseModel
from typing import Annotated, Literal
from annotated_types import Gt

class Fruit(BaseModel):
    name: str
    # 값까지 지정해버린 것.
    # 'red' , 'green'으로 제한.
    color: Literal["red", "green"]
    # "Annotated" 기본 타입에 추가적인 메타데이터나 검증 규칙을 붙이는 문법
    # 실수 타입이어야 하고, "0"이상이어야 함.
    weight: Annotated[float, Gt(0)]
    bazam: dict[str, list[tuple[int, bool, float]]]

print(Fruit(name="Apple", color="red", weight=4.2, bazam={"foobar": [(1, True, 0.1)]}))


name='Apple' color='red' weight=4.2 bazam={'foobar': [(1, True, 0.1)]}


In [4]:
from datetime import datetime
from pydantic import BaseModel

class Meeting(BaseModel):
    when: datetime
    where: bytes
    why: str = "No idea"

m = Meeting(when="2020-01-01T12:00", where="home")

# "model_dump" Pydantic 모델 객체를 일반 Python dict로 바꾸는 메서드
# "exclude_unset" model_dump() 할 때 “사용자가 직접 넣지 않은 기본값 필드는 빼라
print(m.model_dump(exclude_unset=True))
# > {'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}


{'when': datetime.datetime(2020, 1, 1, 12, 0), 'where': b'home'}


In [5]:
from datetime import datetime
from pydantic import BaseModel

class Meeting(BaseModel):
    when: datetime
    where: bytes
    why: str = "No idea"

m = Meeting(when="2020-01-01T12:00", where="home")

# "exclude={\"where\"}" → where 필드는 결과에서 빼라
# "mode" JSON에 넣기 쉬운 값 형태로 변환해서 dict로 내보내라
print(m.model_dump(exclude={"where"}, mode="json"))
# > {'when': '2020-01-01T12:00:00', 'why': 'No idea'}


{'when': '2020-01-01T12:00:00', 'why': 'No idea'}


In [6]:
from datetime import datetime
from pydantic import BaseModel

class Meeting(BaseModel):
    when: datetime
    where: bytes
    why: str = "No idea"

m = Meeting(when="2020-01-01T12:00", where="home")

# "exclude_defaults" 현재 값이 그 필드의 기본값과 같으면 dump 결과에서 빼라
print(m.model_dump_json(exclude_defaults=True))
# > {"when":"2020-01-01T12:00:00","where":"home"}


{"when":"2020-01-01T12:00:00","where":"home"}
